In [ ]:
import numpy as np
import pandas as pd
import os, gc
from tqdm.auto import tqdm
from datetime import datetime, timezone, timedelta

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/한이음/탭별_20년도_1_3월.xlsx", header=None)
df.columns = ['Nan', 'date', 'title', 'description', 'url']
df = df.drop(df.columns[0], axis=1)
df = df.drop(df.index[0], axis=0)
df.head()

,date,title,description,url
1,2020-01-01 23:24:41,[인사] 한국철도시설공단,\n[한국경제TV 전효성 기자]□ 이사대우 임용▲ 기획본부장 신동혁□ 단장 및 본부...,https://n.news.naver.com/mnews/article/215/000...
2,2020-01-01 23:24:24,[인사] 국토교통부,\n[한국경제TV 전효성 기자]□ 과장급 전보▲ 홍보담당관 안경호▲ 녹색도시과장 성...,https://n.news.naver.com/mnews/article/215/000...
3,2020-01-01 23:24:19,[인사] 해외건설협회,\n[한국경제TV 전효성 기자]□ 승진▲ 김성진 회원지원본부장(이사대우)□ 전보▲ ...,https://n.news.naver.com/mnews/article/215/000...
4,2020-01-01 19:00:05,서울 17.6억 대 3.7억…아파트 상·하위 가격차 9년만에 최대,\n[KB국민은행 가격동향 분석]상위 20%가 하위 20%의 6.8배소득보다 자산 ...,https://n.news.naver.com/mnews/article/028/000...
5,2020-01-01 18:16:21,"셈법 복잡해진 은퇴고령자, 상가·꼬마빌딩으로 갈아탄다","\n세금 부담·임대사업 어려워져수익형 부동산으로 선회 가능""임대료 연체 고려 옥석 ...",https://n.news.naver.com/mnews/article/029/000...


In [ ]:
len(df)

1817

In [ ]:
import re
df['description'] = df['description'].apply(lambda x: re.sub(r"\n+", " ", x).strip())

In [ ]:
!git clone https://github.com/rickiepark/nlp-with-transformers.git
%cd nlp-with-transformers
from install import *
install_requirements(chapter=6)

Cloning into 'nlp-with-transformers'...
remote: Enumerating objects: 600, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 600 (delta 12), reused 2 (delta 1), pack-reused 569
Receiving objects: 100% (600/600), 57.83 MiB | 17.30 MiB/s, done.
Resolving deltas: 100% (300/300), done.
/content/nlp-with-transformers
⏳ Installing base requirements ...
✅ Base requirements installed!
Using transformers v4.31.0
Using datasets v2.14.4
Using accelerate v0.21.0
Using sentencepiece v0.1.99
Using sacrebleu v2.3.1
Using rouge_score
Using nltk v3.8.1
Using py7zr v0.20.6


In [ ]:
from transformers import pipeline, set_seed

In [ ]:
import torch
from transformers import PreTrainedTokenizerFast
from transformers import BartForConditionalGeneration

In [ ]:
# @title kobart (ainize)


In [ ]:
from transformers import PreTrainedTokenizerFast, BartForConditionalGeneration
tokenizer = PreTrainedTokenizerFast.from_pretrained("ainize/kobart-news")
model = BartForConditionalGeneration.from_pretrained("ainize/kobart-news")

In [ ]:
df['description'][889]

'[이데일리 황현규 기자] ◇선임 △노동선 부사장 △최종훈 전무(토목사업본부장)황현규 (hhkyu@edaily.co.kr)네이버 홈에서 ‘이데일리’ 뉴스 [구독하기▶] , 청춘뉘우스~ [스냅타임▶]이데일리 기자들의 비밀공간 [기자뉴스룸▶]＜ⓒ종합 경제정보 미디어 이데일리 - 무단전재 & 재배포 금지＞'

In [ ]:
df['description'][398]

'(서울=뉴스1) = ◆국토교통부<전보> ▷실장급 Δ기획조정실장 이문기▷과장급 Δ서울지방국토관리청 관리국장 이호재▶ [ 해피펫 ]  ▶ [터닝 포인트 2020] 구매!▶ 네이버 메인에서 [뉴스1] 구독하기![© 뉴스1코리아(news1.kr), 무단 전재 및 재배포 금지]'

In [ ]:
import random

def generate_summary(news_text):

    deterministic = True

    random.seed(123)
    np.random.seed(123)
    torch.manual_seed(123)
    torch.cuda.manual_seed_all(123)
    if deterministic:
       torch.backends.cudnn.deterministic = True
       torch.backends.cudnn.benchmark = False

    inputs = tokenizer.encode(news_text, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids = model.generate(inputs, max_length=80, min_length=40, num_beams=4, early_stopping=True)
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

df['summary'] = df['description'].apply(generate_summary)

for idx, summary in enumerate(df['summary']):
    print(f"Summary {idx + 1}: {summary}")

 25%|██▍       | 448/1817 [00:00<00:00, 3998.42it/s]

Summary 1: 한국경제TV 이사대우 임용▲ 기획본부장 신동혁등 이사대우 임용▲ 기획본부장 신동혁등 본부장급 전보▲ 자산운영단장 윤여철▲
호남본부장 장형식▲ 강원본부장 김용두□ 처장급 전보▲ 홍보실장 민병창 ▲기획본부 재무전략처장 최근희 ▲ 경영본부 경영노무처장 박진현 ▲ 건설
Summary 2: 과장 과장급 전보▲과장급 전보▲홍보담당관 안경호▲ 녹색도시과장 성호철▲색동도시과장 성호철▲ 지적재조사기획단 사업총괄과장
유승경전효성기자 zeon@wowtv.co.kr.co.kr기자기자 zeon@wowtv.co.kr기자기자 zeon
Summary 3: 201성진 회원지원본부의 김성진 회원지원본부의 전보▲ 정책지원본부장 박형원▲ 정책지원본부장 김운중▲ 기획관리실장 김운중▲
아시아실장 김운중▲ 아시아실장 김태엽(회원지원실장 겸직)▲ 아시아실장 신동우▲ 아·중동실장 신동우▲ 정보화지원실장 이수행(직무대
Summary 4: 고 케이비국민은행의 ‘케이비주택가격동향 시계열 자료’를 보면, 상위 20% 고가 아파트와 하위 20% 저가 아파트의 가격
차이가 9년 만에 최대치로 벌어진 것으로 나타났으며 이는 최근 서울과 일부 광역시에서 고가주택 가격이 급등한 데 반해 중저가 주택이 밀집한 지방
중소도시에선 부동산경기 침체가 이어지면서 아파트값 양극화 현상이 심화한 것으로 풀이
Summary 5: 정부의 부동산대책과 각종 고강도 주택 규제로 세금 부담이 높아지고 임대사업도 어려워지면서 은퇴고령자의 셈법이 복잡해지는
가운데 부동산 업계에서는 늦어도 상반기에는 추가로 기준금리 인하가 이뤄지면 수익형 부동산 인기가 높아질 것으로 내다봤다.
Summary 6: 부동산 전문가들은 경자년 새해에는 9억원 이하 아파트를 중심으로 수요자들이 몰리면서 서울 집값이 오름세를 이어갈 것으로
전망했으며 이에 따라 서울 집값도 9억 이하 아파트들이 밀집한 강북의 노도강(노원구·도봉구·강북구) 지역이 이끌 것으로 예상했으며 올해 서울
집값은 강남보다 강북이 리딩할 것이라고 전망했다.
Summary 7: 호년

100%|██████████| 1817/1817 [00:00<00:00, 4043.12it/s]


Summary 900: 최근 연남동 꼬마빌딩 매매가격이 작년 하락세로 돌아섰고 거래량도 급격히 줄었으며 2017년 74건에서 2018년
53건으로 줄어든 데 이어 2년 연속 감소세를 보이고 있다.
Summary 901: 고가주택에 대한 대출규제를 담은 12.16대책 이후 서울 아파트 시장이 고가 아파트가 밀집한 강남권은 매수심리가
위축되면서 재건축 중심의 하락세를 보였고, 비강남권의 경우 중저가 아파트가 많은 노원, 관악, 도봉구 등에 수요가 유입되면서 집값 오름세를
이어갔으며, 수도권도 강남권 접근성이 개선되는 수원, 용인 등 경기 남부
Summary 902: 12가 아파트 규제가 골자인 '12·16 부동산 대책'이 나온 뒤 오피스텔 투자 수요가 몰릴 것으로 예상됐지만 올해
오피스텔 거래량은 되레 감소한 것으로 나타났는데 수익형 부동산 연구개발기업 상가정보연구소가 국토교통부 실거래가 공개시스템 통계를 분석한 결과
12·16 대책이 나온 뒤 일부 전문가들은 아파트보다 아파트보다 상대적으로 규제에서 자유로운 오피스텔 투자가 증가할 것으로 내다봤지만
Summary 903: 한국0년 공공임대아파트 분양전환가격을 놓고 한국토지주택공사(LH)와 입주민간 분양가 평가방식으로 LH는 ‘감정평가액’을,
주민은 ‘5년 임대방식’ 또는 ‘분양가상한제’를 적용하자는 주장으로 의견 차이를 좁히지 못하고 있어 입주민간 갈등이 거세지고 있다.
Summary 904: 코월 셋째 주에는 청약 작업 이관과 코로나19로 움츠렸던 부동산 시장이 본격적으로 활기를 띨 예정이며 청약접수는 경기 및
강원 두 지역에서 진행되며, 견본주택은 수도권과 지방에서 전국적으로 개관할 예정이며 강원 ‘속초2차 아이파크’는 현장오픈과 유튜브 안내를 동시
진행한다.
Summary 905: 20일에는 과천주공 2단지 주택재건축정비사업 현장이 과천시 복지정책과의 추천을 받아 주거시설 개선 사회공헌 활동을 펼치는
등 2020년에도 롯데건설이 기업의 사회적 책임을 다하고 건강한 시민 사회를 조성하기 위해 부서와 현장에서

In [ ]:
print(df['summary'])

1       한국경제TV 이사대우 임용▲ 기획본부장 신동혁등 이사대우 임용▲ 기획본부장 신동혁등...
2       과장 과장급 전보▲과장급 전보▲홍보담당관 안경호▲ 녹색도시과장 성호철▲색동도시과장 ...
3       201성진 회원지원본부의 김성진 회원지원본부의 전보▲ 정책지원본부장 박형원▲ 정책지...
4       고 케이비국민은행의 ‘케이비주택가격동향 시계열 자료’를 보면, 상위 20% 고가 아...
5       정부의 부동산대책과 각종 고강도 주택 규제로 세금 부담이 높아지고 임대사업도 어려워...
                              ...
1813    20대 국회의원 재산신고 분석 결과 다주택자는 41%인데 비해 무주택자는 9%에 불...
1814    3철도(코레일)가 다음달 1일부터 3개월 간 신종 코로나바이러스 감염증(코로나19)...
1815    IP홈 기술은 물리적으로 거리가 있더라도 IoT 기술을 활용해 가전제품을 다룰 수 ...
1816    코로나19 사태가 장기화되면서 매출이 급감한 중소상인·자영업자·상가임차인들이 생존의...
1817    한국도로공사와 한국철도시설공단이 31일 신종 코로나바이러스 감염증(코로나19)으로 ...
Name: summary, Length: 1817, dtype: object


In [ ]:
# 쓸데없는 내용이 들어있는 인덱스는 삭제

remove = [1, 2, 3, 105, 114, 119, 120, 194, 211, 284, 308, 391, 397, 398, 553, 554, 555, 677, 678, 721, 733, 736, 755, 820, 847, 883, 889, 962, 992, \
          1132, 1134, 1145, 1278, 1279, 1470, 1577, 1738, 1740, 1741, 1743, 1744, 1746]

df.drop(remove, inplace=True)
df.reset_index(drop=True, inplace=True)

print(df)

                     date                                              title  \
0     2020-01-01 19:00:05               서울 17.6억 대 3.7억…아파트 상·하위 가격차 9년만에 최대
1     2020-01-01 18:16:21                      셈법 복잡해진 은퇴고령자, 상가·꼬마빌딩으로 갈아탄다
2     2020-01-01 18:16:19              9억 이하 아파트 대반란 예고… "서울 집값 `노·도·강`이 주도"
3     2020-01-01 18:16:17                 경쟁력 사라진 20·30세대… "저평가지역 적극 공략 바람직"
4     2020-01-01 18:16:15     새 아파트 32만가구 공급 … 6만가구 ↓… 분양물량 서울·경기에 14만여가구 집중
...                   ...                                                ...
1770  2020-03-31 17:50:03                   무주택자 9%, 다주택자 41%…서민과 괴리 큰 국회의원들
1771  2020-03-31 17:48:27                       한국철도, 광명역 도심공항터미널 3개월간 운영 중단
1772  2020-03-31 17:48:04                 IoT 확대하는 스마트홈은 다 좋다?…N번방 같은 해킹 공포도
1773  2020-03-31 17:47:07  임대인 "말도 없이 임대료 안내니.."..임차인 "매출 줄어 깎아달라 해도.." [...
1774  2020-03-31 17:45:41                      도공-철도시설공단, 해외투자개발사업 협력 MOU 체결

                                            description  \
0     [KB국민은행

In [ ]:
## 전처리

import re

def remove_special_characters(text):
    cleaned_text = re.sub(r'[■●▶▲◆△#@\\]', '', text)
    cleaned_text = re.sub(r'\[.*?\]', '', cleaned_text)
    cleaned_text = re.sub(r'\(.*?\)', '', cleaned_text)
    return cleaned_text

df['summary'] = df['summary'].apply(remove_special_characters)

In [ ]:
def remove_newlines(text):
    return text.replace('\n', '')

df['summary'] = df['summary'].apply(remove_newlines)

In [ ]:
df['summary']

0       고 케이비국민은행의 ‘케이비주택가격동향 시계열 자료’를 보면, 상위 20% 고가 아...
1       정부의 부동산대책과 각종 고강도 주택 규제로 세금 부담이 높아지고 임대사업도 어려워...
2       부동산 전문가들은 경자년 새해에는 9억원 이하 아파트를 중심으로 수요자들이 몰리면서...
3       호년건설이 분양예정물량이 더 줄어드는 가운데, 민간택지 분양가 상한제 등의 영향으로...
4       올해 새 아파트 공급예정물량이 지난해보다 더 줄어들면서, '내 집 마련' 경쟁이 한...
                              ...                        
1770    20대 국회의원 재산신고 분석 결과 다주택자는 41%인데 비해 무주택자는 9%에 불...
1771    3철도가 다음달 1일부터 3개월 간 신종 코로나바이러스 감염증 확산 방지와 이용객 ...
1772    IP홈 기술은 물리적으로 거리가 있더라도 IoT 기술을 활용해 가전제품을 다룰 수 ...
1773    코로나19 사태가 장기화되면서 매출이 급감한 중소상인·자영업자·상가임차인들이 생존의...
1774    한국도로공사와 한국철도시설공단이 31일 신종 코로나바이러스 감염증으로 인해 화상으로...
Name: summary, Length: 1775, dtype: object

In [ ]:
df.to_excel("/content/drive/MyDrive/Colab Notebooks/한이음/요약_20년_1_3월.xlsx", sheet_name='Sheet1')

In [ ]:
!pip install openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 1.7 MB/s eta 0:00:00


In [ ]:
import openai
import os

openai.api_key = 'YOUR_KEY'

In [ ]:
def create_sentiment_prompt(text):

    return f"Please analyze the sentiment of the following text and classify it as positive, negative, or neutral: {text}"

def analyze_sentiment(text):

    prompt = create_sentiment_prompt(text)

    response = openai.Completion.create(

        engine="text-davinci-002",

        prompt=prompt,

        max_tokens=50,

        n=1,

        stop=None,

        temperature=0.5,

    )

    sentiment = response.choices[0].text.strip()

    return sentiment

df['sentiment'] = df['summary'].apply(analyze_sentiment)

In [ ]:
df['sentiment']

In [ ]:
def assign_value(sentiment):
    if 'positive' in sentiment:
        return 1
    elif 'negative' in sentiment:
        return -1
    else:
        return 0

df['value'] = df['sentiment'].apply(assign_value)

In [ ]:
df.head()

In [ ]:
## excel로 바꿔서 한이음-데이터-요약-gpt에 저장하면 됨

df.to_excel("C:/Users/82109/Documents/감성분석_22년도_4_6월.xlsx", sheet_name='Sheet1')

In [ ]:
## --------------------------여기까지 --------------

In [ ]:
!pip uninstall nltk

In [ ]:
!pip install nltk

In [ ]:
## BLEU 성능

import nltk.translate.bleu_score as bleu

candidate = """집값 전망 "상승" 35% "유지" 29% "하락" 28%고연령층·새 정부 지지층일수록 집값 하락 전망 지난달 29일 서울의 한 부동산업체 모습. 최근 수도권 부동산 지표는 규제 완화 기대 심리로 반등했다. 뉴시스여론조사업체 한국갤럽이 1일 공개한 여론조사 결과, 집값이 상승할 것이라는 전망이 2020년 6월 조사 이후 2년여 만에 30%대로 하락했다. 새 정부의 부동산 정책에 대한 기대감이 반영된 것인데, 정치 성향과 연령에 따라 입장은 엇갈렸다.한국갤럽이 지난달 29∼31일 성인 유권자 1,001명을 대상으로 조사한 결과(표본오차 95% 신뢰수준에서 ±3.1%포인트)에 따르면, 향후 1년 집값 전망을 묻는 질문에 35%가 \'오른다\'고 답했다. \'변화 없을 것\'이라는 응답은 29%, \'내린다\'는 응답은 28%였다.집값 상승 전망 순지수(상승 전망 비율에서 하락 전망 비율을 뺀 수치)는 직전 조사였던 2021년 9월 조사(43)에서 7로 떨어졌다.\'오른다\'는 응답의 비중이 30%대로 낮아진 것은 2020년 6월 조사(37%) 이후 약 1년 10개월 만이다. 집값 상승 전망은 2020년 7월 조사에서 61%로 치솟은 후 지속적으로 높게 유지돼 왔다. 여론조사 결과는 새 정부 출범 후 집값이 안정화 또는 하향할 것으로 기대하는 여론이 반영된 것으로 보인다. 특히 현 정부를 지지할수록, 고령층일수록 집값이 하락할 것으로 기대하는 태도가 나타난다.응답자 특성별로 집값 상승 전망 순지수를 보면 지역은 대구·경북(-11), 연령은 60대(-4)와 70대 이상(-9), 윤석열 대통령 당선인 국정수행 전망 긍정평가 응답자(-5)가 집값 하락을 전망하는 비중이 우세했다.반면 학생(79)과 윤석열 당선인 국정수행 전망 부정평가 응답자(21), 18∼29세(19) 30대(13) 40대(15) 등은 집값 상승을 전망하는 비중이 높은 것으로 나타났다."본인 소유 집 있어야" 2014년 54% → 2019년 79% 지난달 31일 서울 송파구 롯데월드타워에서 바라본 송파구 일대의 아파트 모습. 고영권 기자본인 소유의 집이 있어야 하는지를 묻는 질문에는 "있어야 한다"는 응답의 비중이 79%로 나타났다. "그럴 필요 없다"는 응답은 19%에 그쳤다. 2014년 7월 조사에서는 \'내 집이 있어야 한다\'가 54%였으나 2017년 1월 63%, 2019년 3월 72%, 2022년 3월 79%로 늘었다.무주택자들은 대체로 이른 시일 안에 본인 소유의 집을 사지는 못할 것으로 내다봤다. 본인 소유 주택 구매 예상 시기를 5년 미만으로 본 비중은 7%, 5∼10년 내는 34%, 10년 이상은 23%였다. \'영영 구매하기 어려울 것\'이라는 응답은 18%였고, \'내 집 마련 의향이 없다\'는 응답은 9%였다.연령별로는 30대(55%)와 40대(42%) 대부분이 5∼10년 내에 집을 구매할 것으로 내다본 반면 20대는 10년 이상(40%)이 소요될 것으로 내다봤다. 60대와 70대 이상은 \'영영 어려울 것 같다\'는 응답의 비중이 각각 48%, 46%로 가장 높았다.경기와 살림살이에 대한 전망에는 개인의 정치 성향이 크게 영향을 미쳤다. 일반적으로 경기 낙관론은 정부 정책 방향에 공감하는 쪽에서 강하고 비관론은 정부에 비판적인 쪽에서 강하다. 정권 교체기가 도래하면서 국민의힘 지지층, 보수 성향 응답자들이 대거 낙관론으로 돌아선 반면 더불어민주당 지지층과 진보 성향 응답자들은 비관적인 태도로 바뀌었다."""
references = [
    "변화 없을 것'이라는 응답은 29%, '내린다'는 응답은 28%였다.집값 상승 전망 순지수(상승 전망 비율에서 하락 전망 비율을 뺀 수치)는 직전 조사였던 2021년 9월 조사(43)에서 7로 떨어졌다.'오른다'는 응답의 비중이 30%대로 낮아진 것은 2020년 6월 조사(37%) 이후 약 1년 10개월 만이다."]

print('BLEU:', bleu.sentence_bleu([list(map(lambda ref: ref.split(), references))], candidate.split()))